# Classificação de Imagens por CNN

Este notebook foi organizado em seções separadas por arquitetura de CNN para facilitar a execução no Google Colab.

Modelos incluídos:

- DenseNet121: baseline do artigo original
- ResNet50: CNN clássica e muito citada
- ConvNeXt V2: CNN moderna
- InceptionV3: arquitetura com conceito diferente

A ideia é manter a base comum no início e deixar cada modelo em um bloco próprio.

In [3]:
# Base comum para todos os experimentos

import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Download da base
path = kagglehub.dataset_download("mahyeks/almond-varieties")
dataset_dir = os.path.join(path, "dataset")
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

classes = sorted(
    nome for nome in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, nome))
)

image_paths = []
labels = []
for classe in classes:
    classe_dir = os.path.join(dataset_dir, classe)
    for nome_arquivo in sorted(os.listdir(classe_dir)):
        if nome_arquivo.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(classe_dir, nome_arquivo))
            labels.append(classe)

base_df = pd.DataFrame({"image_path": image_paths, "label": labels})
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(base_df["label"])
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    base_df["image_path"],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Base carregada com sucesso.")
print(base_df["label"].value_counts())

100%|██████████| 96.5M/96.5M [00:00<00:00, 105MB/s] 

Extracting files...


Base carregada com sucesso.
label
KAPADOKYA    465
AK           401
SIRA         384
NURLU        306
Name: count, dtype: int64


## DenseNet121

Baseline do artigo original. Use este bloco para a primeira comparação.

In [ ]:
# DenseNet121 - classificação com fine-tuning leve

from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def carregar_imagens(lista_paths, target_size=(224, 224)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


X_train = carregar_imagens(X_train_paths.tolist())
X_test = carregar_imagens(X_test_paths.tolist())

base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento sugerido no Colab
history = model.fit(
     X_train,
     y_train,
     validation_split=0.2,
     epochs=10,
     batch_size=32,
     verbose=1,
 )

# Avaliação
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## ResNet50

CNN clássica e muito citada. Use este bloco para o segundo experimento.

In [ ]:
# ResNet50 - CNN clássica e muito citada

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def carregar_imagens_resnet(lista_paths, target_size=(224, 224)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


X_train_resnet = carregar_imagens_resnet(X_train_paths.tolist())
X_test_resnet = carregar_imagens_resnet(X_test_paths.tolist())

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_resnet = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model_resnet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento sugerido no Colab
history = model_resnet.fit(
     X_train_resnet,
     y_train,
     validation_split=0.2,
     epochs=10,
     batch_size=32,
     verbose=1,
 )

# Avaliação
y_pred = np.argmax(model_resnet.predict(X_test_resnet, verbose=0), axis=1)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## ConvNeXt V2

CNN moderna. Use este bloco para o terceiro experimento.

In [ ]:
# ConvNeXt V2 - CNN moderna

# No Colab, use `timm` e `torch` para uma implementação próxima da ConvNeXt V2.
# pip install timm torch torchvision

import torch
import timm
from PIL import Image
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


device = "cuda" if torch.cuda.is_available() else "cpu"

feature_extractor_convnext = timm.create_model("convnextv2_tiny", pretrained=True, num_classes=0).to(device)
feature_extractor_convnext.eval()
transform_convnext = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def carregar_imagens_convnext(lista_paths):
    imagens = []
    for image_path in lista_paths:
        img = Image.open(image_path).convert("RGB")
        imagens.append(transform_convnext(img))
    batch_tensor = torch.stack(imagens).to(device)
    with torch.no_grad():
        features = feature_extractor_convnext(batch_tensor).cpu().numpy()
    return features


X_train_convnext = carregar_imagens_convnext(X_train_paths.tolist())
X_test_convnext = carregar_imagens_convnext(X_test_paths.tolist())

scaler_convnext = StandardScaler()
X_train_convnext = scaler_convnext.fit_transform(X_train_convnext)
X_test_convnext = scaler_convnext.transform(X_test_convnext)

# Treinamento sugerido no Colab
svm = SVC(kernel="linear", C=1.0)
svm.fit(X_train_convnext, y_train)
y_pred = svm.predict(X_test_convnext)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## InceptionV3

Arquitetura com conceito diferente. Use este bloco para o quarto experimento.

In [4]:
# InceptionV3 - modelo de arquitetura diferente

from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def carregar_imagens_inception(lista_paths, target_size=(299, 299)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


X_train_inception = carregar_imagens_inception(X_train_paths.tolist())
X_test_inception = carregar_imagens_inception(X_test_paths.tolist())

base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_inception = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model_inception.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento sugerido no Colab
history = model_inception.fit(
     X_train_inception,
     y_train,
     validation_split=0.2,
     epochs=10,
     batch_size=32,
     verbose=1,
 )

# Avaliação
y_pred = np.argmax(model_inception.predict(X_test_inception, verbose=0), axis=1)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
12/32 ━━━━━━━━━━━━━━━━━━━━ 2:18 7s/step - accuracy: 0.2146 - loss: 1.6229

KeyboardInterrupt: 